In [0]:
customers_path="/Volumes/workspace/default/servicetrack_raw/customers.csv"
devices_path="/Volumes/workspace/default/servicetrack_raw/devices.csv"
service_jobs_path="/Volumes/workspace/default/servicetrack_raw/service_jobs.csv"

customers_df=spark.read.option("header",True).option("inferSchema",True).csv(customers_path)
devices_df=spark.read.option("header",True).option("inferSchema",True).csv(devices_path)
service_jobs_df=spark.read.option("header",True).option("inferSchema",True).csv(service_jobs_path)

print("Customers:",customers_df.count())
print("Devices:",devices_df.count())
print("Service Jobs:",service_jobs_df.count())

Customers: 300
Devices: 43
Service Jobs: 1510


In [0]:
customers_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_customers")
devices_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_devices")
service_jobs_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_service_jobs")

print("Bronze tables created successfully")

Bronze tables created successfully


In [0]:
print("Customers:",spark.table("workspace.default.bronze_customers").count())
print("Devices:",spark.table("workspace.default.bronze_devices").count())
print("Service Jobs:",spark.table("workspace.default.bronze_service_jobs").count())

print("\nCustomers Schema:")
spark.table("workspace.default.bronze_customers").printSchema()

print("\nDevices Schema:")
spark.table("workspace.default.bronze_devices").printSchema()

print("\nService Jobs Schema:")
spark.table("workspace.default.bronze_service_jobs").printSchema()

Customers: 300
Devices: 43
Service Jobs: 1510

Customers Schema:
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- phone_number: long (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = true)


Devices Schema:
root
 |-- device_id: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- model_series: string (nullable = true)
 |-- warranty_months: integer (nullable = true)
 |-- price_range: string (nullable = true)


Service Jobs Schema:
root
 |-- job_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- issue_type: string (nullable = true)
 |-- job_status: string (nullable = true)
 |-- received_date: date (nullable = true)
 |-- promised_date: date (nullable = true)
 |-- completed_date: date (nullable = true)
 |-- technician_id: string (nullable 

In [0]:
service_jobs_df.printSchema()
display(service_jobs_df.limit(10))

root
 |-- job_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- issue_type: string (nullable = true)
 |-- job_status: string (nullable = true)
 |-- received_date: date (nullable = true)
 |-- promised_date: date (nullable = true)
 |-- completed_date: date (nullable = true)
 |-- technician_id: string (nullable = true)
 |-- technician_name: string (nullable = true)
 |-- repair_notes: string (nullable = true)
 |-- estimated_cost: double (nullable = true)
 |-- actual_cost: double (nullable = true)



job_id,customer_id,device_id,issue_type,job_status,received_date,promised_date,completed_date,technician_id,technician_name,repair_notes,estimated_cost,actual_cost
JOB00030,CUST0055,DEV036,Battery Issue,Pending,2024-02-13,2024-02-18,null,T005,Kavitha Nair,null,7662.33,null
JOB01498,CUST0145,DEV037,Motherboard Failure,Completed,2024-02-06,2024-02-11,2024-02-13,T008,Arjun Iyer,null,7751.72,4258.76
JOB01443,CUST0046,DEV027,Motherboard Failure,Completed,2024-03-01,2024-03-06,2024-03-04,T002,Suresh Rao,Customer informed,6912.68,1558.22
JOB00945,CUST0161,DEV018,Water Damage,Completed,2024-03-03,2024-03-08,2024-03-06,T001,Rajesh Kumar,Under warranty,605.69,2263.41
JOB00957,CUST0129,DEV019,RAM Issue,Completed,2024-01-09,2024-01-14,2024-01-10,T003,Priya Singh,Customer informed,5034.79,7386.68
JOB01455,CUST0192,DEV018,Keyboard Fault,Pending,2024-02-19,2024-02-24,null,T008,Arjun Iyer,Customer follow-up pending,4095.16,null
JOB00104,CUST0298,DEV035,Motherboard Failure,Completed,2024-01-18,2024-01-23,2024-01-24,T004,Amit Patel,null,7294.59,3956.07
JOB00294,CUST0022,DEV038,Charging Port Fault,Completed,2024-01-22,2024-01-27,2024-01-26,T005,Kavitha Nair,Under warranty,899.25,6435.76
JOB01400,CUST0036,DEV030,Hinge Broken,Cancelled,2024-01-28,2024-02-02,2024-01-31,T005,Kavitha Nair,null,2197.06,null
JOB01090,CUST0201,DEV017,Overheating,Completed,2024-03-04,2024-03-09,2024-03-10,T003,Priya Singh,Returned to customer,3706.98,7369.66


In [0]:
print(service_jobs_df.columns)

['job_id', 'customer_id', 'device_id', 'issue_type', 'job_status', 'received_date', 'promised_date', 'completed_date', 'technician_id', 'technician_name', 'repair_notes', 'estimated_cost', 'actual_cost']


In [0]:
from pyspark.sql.functions import col,trim,coalesce,lit,to_date,datediff

jobs=spark.table("workspace.default.bronze_service_jobs")
customers=spark.table("workspace.default.bronze_customers")
devices=spark.table("workspace.default.bronze_devices")

jobs_clean=(
    jobs
    .dropDuplicates(["job_id"])
    .withColumn("technician_name",coalesce(trim(col("technician_name")),lit("Unknown")))
    .withColumn("received_date",to_date(col("received_date")))
    .withColumn("promised_date",to_date(col("promised_date")))
    .withColumn("completed_date",to_date(col("completed_date")))
    .withColumn("repair_duration_days",
                datediff(col("completed_date"),col("received_date")))
    .withColumn("delay_days",
                datediff(col("completed_date"),col("promised_date")))
)

silver_df=(
    jobs_clean
    .join(customers,"customer_id","left")
    .join(devices,"device_id","left")
)

silver_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.silver_service_jobs"
)

print("Silver records:",silver_df.count())
print("Silver table created successfully")

Silver records: 1500
Silver table created successfully


In [0]:
from pyspark.sql.functions import col,trim

print("Silver records:",silver_df.count())
print("Duplicate job IDs:",silver_df.groupBy("job_id").count().filter(col("count")>1).count())
print("Missing technician names:",silver_df.filter(col("technician_name").isNull()| (trim(col("technician_name"))=="")).count())
print("Incomplete jobs:",silver_df.filter(col("completed_date").isNull()).count())
print("Negative repair durations:",silver_df.filter(col("repair_duration_days")<0).count())
print("Negative delays:",silver_df.filter(col("delay_days")<0).count())

Silver records: 1500
Duplicate job IDs: 0
Missing technician names: 0
Incomplete jobs: 370
Negative repair durations: 0
Negative delays: 664


In [0]:
from pyspark.sql.functions import col,when

gold_turnaround=(
    silver_df
    .select(
        "job_id",
        "customer_id",
        "device_id",
        "issue_type",
        "job_status",
        "received_date",
        "promised_date",
        "completed_date",
        "repair_duration_days",
        "delay_days"
    )
    .withColumn(
        "delivery_status",
        when(col("completed_date").isNull(),"Incomplete")
        .when(col("delay_days")>0,"Delayed")
        .otherwise("On Time")
    )
)

gold_turnaround.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_repair_turnaround"
)

print("Gold Repair Turnaround records:",gold_turnaround.count())
print("Gold Repair Turnaround table created successfully")

Gold Repair Turnaround records: 1500
Gold Repair Turnaround table created successfully


In [0]:
from pyspark.sql.functions import col,count,avg,round

gold_technician=(
    silver_df
    .groupBy("technician_id","technician_name")
    .agg(
        count("job_id").alias("total_jobs"),
        count(when(col("job_status")=="Completed",1)).alias("completed_jobs"),
        round(avg("repair_duration_days"),2).alias("avg_repair_duration_days"),
        round(avg("delay_days"),2).alias("avg_delay_days"),
        round(avg("actual_cost"),2).alias("avg_actual_cost")
    )
)

gold_technician.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_technician_performance"
)

print("Gold Technician records:",gold_technician.count())
print("Gold Technician Performance table created successfully")

Gold Technician records: 16
Gold Technician Performance table created successfully


In [0]:
from pyspark.sql.functions import col,count

gold_repeat_customers=(
    silver_df
    .groupBy("customer_id","customer_name")
    .agg(count("job_id").alias("total_jobs"))
    .filter(col("total_jobs")>1)
)

gold_repeat_customers.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_repeat_customers"
)

print("Gold Repeat Customer records:",gold_repeat_customers.count())
print("Gold Repeat Customer table created successfully")

Gold Repeat Customer records: 271
Gold Repeat Customer table created successfully


In [0]:
from pyspark.sql.functions import col,count

gold_device_failures=(
    silver_df
    .groupBy("device_id","brand","device_type","model_series")
    .agg(count("job_id").alias("total_jobs"))
    .orderBy(col("total_jobs").desc())
)

gold_device_failures.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_device_failures"
)

print("Gold Device Failure records:",gold_device_failures.count())
print("Gold Device Failure table created successfully")

Gold Device Failure records: 43
Gold Device Failure table created successfully


In [0]:
from pyspark.sql.functions import col,count,avg,round

gold_issue_analysis=(
    silver_df
    .groupBy("issue_type")
    .agg(
        count("job_id").alias("total_jobs"),
        count(when(col("job_status")=="Completed",1)).alias("completed_jobs"),
        round(avg("repair_duration_days"),2).alias("avg_repair_duration_days"),
        round(avg("delay_days"),2).alias("avg_delay_days"),
        round(avg("actual_cost"),2).alias("avg_actual_cost")
    )
    .orderBy(col("total_jobs").desc())
)

gold_issue_analysis.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_issue_analysis"
)

print("Gold Issue Analysis records:",gold_issue_analysis.count())
print("Gold Issue Analysis table created successfully")

Gold Issue Analysis records: 15
Gold Issue Analysis table created successfully


In [0]:
gold_issue_analysis.select(
    "issue_type",
    "total_jobs",
    "completed_jobs",
    "avg_repair_duration_days",
    "avg_delay_days",
    "avg_actual_cost"
).show(15,truncate=False)

+--------------------+----------+--------------+------------------------+--------------+---------------+
|issue_type          |total_jobs|completed_jobs|avg_repair_duration_days|avg_delay_days|avg_actual_cost|
+--------------------+----------+--------------+------------------------+--------------+---------------+
|Charging Port Fault |124       |85            |4.62                    |-0.38         |4717.06        |
|Wi-Fi Not Connecting|121       |85            |4.55                    |-0.45         |4559.59        |
|Software Crash      |113       |87            |4.66                    |-0.34         |4795.98        |
|Water Damage        |110       |75            |4.33                    |-0.68         |4531.35        |
|Keyboard Fault      |106       |71            |4.21                    |-0.79         |4790.98        |
|Hard Disk Failure   |105       |73            |4.5                     |-0.5          |4221.96        |
|Screen Damage       |101       |70            |4.66   

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col,row_number

window_spec=Window.partitionBy("customer_id").orderBy(col("received_date").desc())

gold_customer_visit_history=(
    silver_df
    .withColumn("row_num",row_number().over(window_spec))
    .filter(col("row_num")==1)
    .drop("row_num")
)

gold_customer_visit_history.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_customer_visit_history"
)

print("Gold Customer Visit History records:",gold_customer_visit_history.count())
print("Gold Customer Visit History table created successfully")

Gold Customer Visit History records: 293
Gold Customer Visit History table created successfully


In [0]:
print("Gold Customer Visit History:",gold_customer_visit_history.count())
print("Gold Technician Performance:",gold_technician.count())
print("Gold Repair Turnaround:",gold_turnaround.count())
print("Gold Repeat Customers:",gold_repeat_customers.count())
print("Gold Device Failures:",gold_device_failures.count())
print("Gold Issue Analysis:",gold_issue_analysis.count())

Gold Customer Visit History: 293
Gold Technician Performance: 16
Gold Repair Turnaround: 1500
Gold Repeat Customers: 271
Gold Device Failures: 43
Gold Issue Analysis: 15


In [0]:
gold_technician.select(
    "technician_id",
    "technician_name",
    "total_jobs",
    "completed_jobs",
    "avg_repair_duration_days",
    "avg_delay_days",
    "avg_actual_cost"
).show(16,truncate=False)

+-------------+---------------+----------+--------------+------------------------+--------------+---------------+
|technician_id|technician_name|total_jobs|completed_jobs|avg_repair_duration_days|avg_delay_days|avg_actual_cost|
+-------------+---------------+----------+--------------+------------------------+--------------+---------------+
|T005         |Kavitha Nair   |182       |137           |2.57                    |-2.43         |4361.11        |
|T006         |Unknown        |8         |6             |6.0                     |1.0           |3229.36        |
|T007         |Meena Reddy    |199       |146           |4.17                    |-0.83         |4620.49        |
|T008         |Arjun Iyer     |152       |107           |7.11                    |2.11          |4769.29        |
|T004         |Unknown        |10        |7             |6.13                    |1.13          |2755.79        |
|T003         |Priya Singh    |203       |135           |3.55                    |-1.45 

In [0]:
%sql
SELECT
    job_id,
    customer_id,
    customer_name,
    issue_type,
    job_status,
    received_date,
    promised_date,
    completed_date,
    CASE
        WHEN completed_date IS NULL THEN 'Incomplete'
        WHEN completed_date > promised_date THEN 'Delayed'
        ELSE 'On Time'
    END AS delivery_status
FROM workspace.default.silver_service_jobs
ORDER BY received_date DESC;

job_id,customer_id,customer_name,issue_type,job_status,received_date,promised_date,completed_date,delivery_status
JOB00467,CUST0064,Imran Singh,Power Not Turning On,In Progress,2024-03-21,2024-03-26,null,Incomplete
JOB00532,CUST0125,Ajay Ali,Water Damage,Completed,2024-03-21,2024-03-26,2024-03-23,On Time
JOB00916,CUST0250,Harish Chaudhary,Wi-Fi Not Connecting,Completed,2024-03-21,2024-03-26,2024-03-23,On Time
JOB00442,CUST0125,Ajay Ali,Screen Damage,Completed,2024-03-21,2024-03-26,2024-03-25,On Time
JOB01133,CUST0285,Pooja Iyer,Wi-Fi Not Connecting,Completed,2024-03-21,2024-03-26,2024-03-24,On Time
JOB00101,CUST0121,Nisha Bajaj,Camera Not Working,Completed,2024-03-21,2024-03-26,2024-03-24,On Time
JOB01301,CUST0218,Deepika Patel,Camera Not Working,Completed,2024-03-21,2024-03-26,2024-03-29,Delayed
JOB00627,CUST0183,Lakshmi Khan,Battery Issue,Completed,2024-03-21,2024-03-26,2024-03-23,On Time
JOB01310,CUST0082,Madhu Verma,Motherboard Failure,In Progress,2024-03-21,2024-03-26,null,Incomplete
JOB01093,CUST0214,Priya Iyer,RAM Issue,Completed,2024-03-21,2024-03-26,2024-03-23,On Time


In [0]:
%sql
SELECT
    issue_type,
    COUNT(*) AS total_jobs,
    COUNT(CASE WHEN job_status = 'Completed' THEN 1 END) AS completed_jobs
FROM workspace.default.silver_service_jobs
GROUP BY issue_type
HAVING COUNT(*) > 50
ORDER BY total_jobs DESC;

issue_type,total_jobs,completed_jobs
Charging Port Fault,124,85
Wi-Fi Not Connecting,121,85
Software Crash,113,87
Water Damage,110,75
Keyboard Fault,106,71
Hard Disk Failure,105,73
Screen Damage,101,70
Overheating,100,71
Motherboard Failure,95,63
Battery Issue,94,67


In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    COUNT(*) AS total_jobs,
    SUM(CASE WHEN job_status = 'Completed' THEN 1 ELSE 0 END) AS completed_jobs
FROM workspace.default.silver_service_jobs
GROUP BY customer_id, customer_name
HAVING COUNT(*) > 3
ORDER BY total_jobs DESC;

customer_id,customer_name,total_jobs,completed_jobs
CUST0300,Chandana Shetty,14,9
CUST0231,Neha Fernandez,14,10
CUST0052,Ashok Bhat,14,9
CUST0146,Prakash Shetty,14,7
CUST0003,Bhavna Tiwari,14,9
CUST0291,Rohan Bhat,14,10
CUST0192,Sonam Iyer,13,7
CUST0035,Gaurav Chaudhary,13,10
CUST0274,Vimala Qureshi,13,12
CUST0183,Lakshmi Khan,13,11


In [0]:
%sql
SELECT
    job_id,
    customer_id,
    customer_name,
    issue_type,
    job_status,
    received_date,
    ROW_NUMBER() OVER(
        PARTITION BY customer_id
        ORDER BY received_date DESC
    ) AS visit_rank
FROM workspace.default.silver_service_jobs
ORDER BY customer_id, visit_rank;

job_id,customer_id,customer_name,issue_type,job_status,received_date,visit_rank
JOB00991,CUST0001,Kiran Patel,Software Crash,Cancelled,2024-03-13,1
JOB01006,CUST0001,Kiran Patel,RAM Issue,Completed,2024-03-04,2
JOB01123,CUST0001,Kiran Patel,Overheating,Completed,2024-02-02,3
JOB01328,CUST0001,Kiran Patel,Charging Port Fault,Pending,2024-01-06,4
JOB00107,CUST0003,Bhavna Tiwari,Screen Damage,Completed,2024-03-18,1
JOB00656,CUST0003,Bhavna Tiwari,RAM Issue,Completed,2024-03-13,2
JOB00517,CUST0003,Bhavna Tiwari,Overheating,Cancelled,2024-03-12,3
JOB00381,CUST0003,Bhavna Tiwari,Wi-Fi Not Connecting,Cancelled,2024-03-06,4
JOB01232,CUST0003,Bhavna Tiwari,Water Damage,Cancelled,2024-03-04,5
JOB00024,CUST0003,Bhavna Tiwari,Power Not Turning On,Pending,2024-03-01,6


In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    job_id,
    issue_type,
    job_status,
    received_date,
    promised_date,
    completed_date
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER(
            PARTITION BY customer_id
            ORDER BY received_date DESC
        ) AS rn
    FROM workspace.default.silver_service_jobs
)
WHERE rn = 1
ORDER BY received_date DESC;

customer_id,customer_name,job_id,issue_type,job_status,received_date,promised_date,completed_date
CUST0285,Pooja Iyer,JOB01133,Wi-Fi Not Connecting,Completed,2024-03-21,2024-03-26,2024-03-24
CUST0082,Madhu Verma,JOB01310,Motherboard Failure,In Progress,2024-03-21,2024-03-26,null
CUST0121,Nisha Bajaj,JOB00101,Camera Not Working,Completed,2024-03-21,2024-03-26,2024-03-24
CUST0099,Tanvi Iyer,JOB00655,Wi-Fi Not Connecting,Cancelled,2024-03-21,2024-03-26,2024-03-24
CUST0183,Lakshmi Khan,JOB00627,Battery Issue,Completed,2024-03-21,2024-03-26,2024-03-23
CUST0156,Simran Mehta,JOB00988,Software Crash,Completed,2024-03-21,2024-03-26,2024-03-25
CUST0269,Prakash Malhotra,JOB01233,Overheating,In Progress,2024-03-21,2024-03-26,null
CUST0250,Harish Chaudhary,JOB00916,Wi-Fi Not Connecting,Completed,2024-03-21,2024-03-26,2024-03-23
CUST0276,Vinay Dubey,JOB00392,Speaker Problem,In Progress,2024-03-21,2024-03-26,null
CUST0094,Sanjay Bajaj,JOB01480,Power Not Turning On,Completed,2024-03-21,2024-03-26,2024-03-31


In [0]:
%sql
SELECT
    issue_type,
    COUNT(*) AS total_jobs,
    SUM(CASE WHEN job_status = 'Completed' THEN 1 ELSE 0 END) AS completed_jobs,
    ROUND(AVG(CASE WHEN completed_date IS NOT NULL
        THEN DATEDIFF(completed_date, received_date) END),2) AS avg_repair_days
FROM workspace.default.silver_service_jobs
GROUP BY issue_type
ORDER BY total_jobs DESC;

issue_type,total_jobs,completed_jobs,avg_repair_days
Charging Port Fault,124,85,4.62
Wi-Fi Not Connecting,121,85,4.55
Software Crash,113,87,4.66
Water Damage,110,75,4.33
Keyboard Fault,106,71,4.21
Hard Disk Failure,105,73,4.5
Screen Damage,101,70,4.66
Overheating,100,71,4.61
Motherboard Failure,95,63,4.14
Battery Issue,94,67,4.27


In [0]:
%sql
SELECT
    technician_id,
    technician_name,
    COUNT(*) AS total_jobs,
    SUM(CASE WHEN job_status='Completed' THEN 1 ELSE 0 END) AS completed_jobs,
    ROUND(AVG(repair_duration_days),2) AS avg_repair_days,
    ROUND(AVG(delay_days),2) AS avg_delay_days,
    ROUND(AVG(actual_cost),2) AS avg_actual_cost
FROM workspace.default.silver_service_jobs
GROUP BY technician_id, technician_name
ORDER BY total_jobs DESC;

technician_id,technician_name,total_jobs,completed_jobs,avg_repair_days,avg_delay_days,avg_actual_cost
T003,Priya Singh,203,135,3.55,-1.45,4591.74
T007,Meena Reddy,199,146,4.17,-0.83,4620.49
T005,Kavitha Nair,182,137,2.57,-2.43,4361.11
T002,Suresh Rao,182,115,4.3,-0.7,4347.28
T006,Deepak Sharma,178,129,5.21,0.21,4548.69
T001,Rajesh Kumar,175,127,2.98,-2.02,4518.25
T004,Amit Patel,154,112,6.48,1.48,4475.13
T008,Arjun Iyer,152,107,7.11,2.11,4769.29
T002,Unknown,14,7,4.75,-0.25,3348.58
T005,Unknown,10,6,2.5,-2.5,3740.73


In [0]:
%sql
SELECT
    technician_id,
    technician_name,
    total_jobs,
    completed_jobs,
    ROUND(completed_jobs * 100.0 / total_jobs,2) AS completion_rate,
    avg_repair_days,
    avg_delay_days,
    avg_actual_cost
FROM (
    SELECT
        technician_id,
        technician_name,
        COUNT(*) AS total_jobs,
        SUM(CASE WHEN job_status='Completed' THEN 1 ELSE 0 END) AS completed_jobs,
        ROUND(AVG(repair_duration_days),2) AS avg_repair_days,
        ROUND(AVG(delay_days),2) AS avg_delay_days,
        ROUND(AVG(actual_cost),2) AS avg_actual_cost
    FROM workspace.default.silver_service_jobs
    GROUP BY technician_id, technician_name
)
ORDER BY completion_rate DESC;

technician_id,technician_name,total_jobs,completed_jobs,completion_rate,avg_repair_days,avg_delay_days,avg_actual_cost
T005,Kavitha Nair,182,137,75.27,2.57,-2.43,4361.11
T006,Unknown,8,6,75.00,6.0,1.0,3229.36
T007,Meena Reddy,199,146,73.37,4.17,-0.83,4620.49
T004,Amit Patel,154,112,72.73,6.48,1.48,4475.13
T001,Rajesh Kumar,175,127,72.57,2.98,-2.02,4518.25
T006,Deepak Sharma,178,129,72.47,5.21,0.21,4548.69
T001,Unknown,7,5,71.43,3.0,-2.0,2964.74
T008,Arjun Iyer,152,107,70.39,7.11,2.11,4769.29
T004,Unknown,10,7,70.00,6.13,1.13,2755.79
T003,Priya Singh,203,135,66.50,3.55,-1.45,4591.74


In [0]:
from pyspark.sql.functions import col

top_technicians=gold_technician.filter(col("total_jobs")>=150).withColumn("completion_rate",round(col("completed_jobs")*100.0/col("total_jobs"),2)).orderBy(col("completion_rate").desc())

top_technicians.show()

+-------------+---------------+----------+--------------+------------------------+--------------+---------------+---------------+
|technician_id|technician_name|total_jobs|completed_jobs|avg_repair_duration_days|avg_delay_days|avg_actual_cost|completion_rate|
+-------------+---------------+----------+--------------+------------------------+--------------+---------------+---------------+
|         T005|   Kavitha Nair|       182|           137|                    2.57|         -2.43|        4361.11|          75.27|
|         T007|    Meena Reddy|       199|           146|                    4.17|         -0.83|        4620.49|          73.37|
|         T004|     Amit Patel|       154|           112|                    6.48|          1.48|        4475.13|          72.73|
|         T001|   Rajesh Kumar|       175|           127|                    2.98|         -2.02|        4518.25|          72.57|
|         T006|  Deepak Sharma|       178|           129|                    5.21|        

In [0]:
top_technicians.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_top_technicians")
print("Gold Top Technicians records:",top_technicians.count())
print("Gold Top Technicians table created successfully")

Gold Top Technicians records: 8
Gold Top Technicians table created successfully


In [0]:
tables=spark.sql("SHOW TABLES IN workspace.default")
tables.filter(col("tableName").startswith("gold_")).show(truncate=False)

+--------+---------------------------+-----------+
|database|tableName                  |isTemporary|
+--------+---------------------------+-----------+
|default |gold_customer_visit_history|false      |
|default |gold_device_failures       |false      |
|default |gold_issue_analysis        |false      |
|default |gold_repair_turnaround     |false      |
|default |gold_repeat_customers      |false      |
|default |gold_technician_performance|false      |
|default |gold_top_technicians       |false      |
+--------+---------------------------+-----------+



In [0]:
print("========== SERVICETRACK PIPELINE SUMMARY ==========")
print("Bronze Customers:",customers_df.count())
print("Bronze Devices:",devices_df.count())
print("Bronze Service Jobs:",service_jobs_df.count())
print("Silver Service Jobs:",silver_df.count())
print("Gold Customer Visit History:",gold_customer_visit_history.count())
print("Gold Device Failures:",gold_device_failures.count())
print("Gold Issue Analysis:",gold_issue_analysis.count())
print("Gold Repair Turnaround:",gold_turnaround.count())
print("Gold Repeat Customers:",gold_repeat_customers.count())
print("Gold Technician Performance:",gold_technician.count())
print("Gold Top Technicians:",top_technicians.count())
print("===================================================")
print("ServiceTrack pipeline completed successfully!")

========== SERVICETRACK PIPELINE SUMMARY ==========
Bronze Customers: 300
Bronze Devices: 43
Bronze Service Jobs: 1510
Silver Service Jobs: 1500
Gold Customer Visit History: 293
Gold Device Failures: 43
Gold Issue Analysis: 15
Gold Repair Turnaround: 1500
Gold Repeat Customers: 271
Gold Technician Performance: 16
Gold Top Technicians: 8
ServiceTrack pipeline completed successfully!
